# Part 1: Baseline CLIP Embeddings for ImageNet Val

**Goal:** Compute deterministic CLIP B/32 embeddings for a sample of ImageNet validation images.

Pipeline:
1. Preview data — folder structure, sample images, label mapping
2. Load model — CLIP ViT-B/32 via open_clip
3. Compute embeddings in batches — save after each batch
4. Analysis — PCA colored by class, intra/inter-class similarity

In [ ]:
import json, os, time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader, Dataset

# Resolve repo root
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "part1_baseline_embeddings" else Path.cwd()
if not (PROJECT_ROOT / "round3").exists() and len(PROJECT_ROOT.parents) >= 2 and (PROJECT_ROOT.parents[1] / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass(frozen=True)
class ModelSpec:
    key: str
    open_clip_model: str
    open_clip_pretrained: str


MODEL_REGISTRY: Dict[str, ModelSpec] = {
    "clip_b32": ModelSpec("clip_b32", "ViT-B-32", "openai"),
    "clip_l14": ModelSpec("clip_l14", "ViT-L-14", "openai"),
    "pe_core_b16": ModelSpec("pe_core_b16", "PE-Core-B-16", "meta"),
    "siglip2_b16": ModelSpec("siglip2_b16", "ViT-B-16-SigLIP2", "webli"),
    "siglip2_so400m": ModelSpec("siglip2_so400m", "ViT-SO400M-14-SigLIP2-378", "webli"),
    "siglip2_g16": ModelSpec("siglip2_g16", "ViT-gopt-16-SigLIP2-384", "webli"),
}


class ImagePathDataset(Dataset):
    def __init__(self, image_paths: Sequence[str]) -> None:
        self.image_paths = [str(Path(p)) for p in image_paths]

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, str, str]:
        path = self.image_paths[idx]
        with Image.open(path) as img:
            image = img.convert("RGB")
        class_name = Path(path).parent.name
        return image, path, class_name


def pil_collate(batch: Sequence[Tuple[Image.Image, str, str]]) -> Tuple[List[Image.Image], List[str], List[str]]:
    images = [item[0] for item in batch]
    paths = [item[1] for item in batch]
    class_names = [item[2] for item in batch]
    return images, paths, class_names


def resolve_project_root(cwd: Path) -> Path:
    project_root = cwd.parents[1] if cwd.name == "part1_baseline_embeddings" else cwd
    if (project_root / "round3").exists():
        return project_root
    if len(project_root.parents) >= 2 and (project_root.parents[1] / "data").exists():
        return project_root.parents[1]
    return project_root


def detect_best_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def resolve_device(requested: str = "auto") -> str:
    if requested.strip().lower() in {"", "auto"}:
        return detect_best_device()
    return requested


def default_num_workers() -> int:
    try:
        from IPython import get_ipython

        if get_ipython() is not None:
            return 0
    except Exception:
        pass
    cpu_count = os.cpu_count() or 4
    return max(2, min(8, cpu_count // 2))


def default_batch_size(device: str) -> int:
    if device == "cpu":
        return 32
    if device in {"mps", "cuda"}:
        return 256
    return 64


def use_fp16_for(device: str, requested: bool = True) -> bool:
    return bool(requested and device in {"mps", "cuda"})


def list_images(data_dir: str) -> List[str]:
    root = Path(data_dir)
    if not root.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")
    paths = [str(p) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
    paths.sort()
    if not paths:
        raise ValueError(f"No image files found under {data_dir}")
    return paths


def sample_paths(paths: Sequence[str], num_images: int, seed: int) -> List[str]:
    path_list = list(paths)
    if num_images <= 0 or num_images >= len(path_list):
        return path_list
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(path_list), size=num_images, replace=False))
    return [path_list[i] for i in idx]


def build_loader(
    image_paths: Sequence[str],
    batch_size: int,
    num_workers: int,
    persistent_workers: Optional[bool] = None,
    prefetch_factor: Optional[int] = None,
    pin_memory: bool = True,
) -> DataLoader:
    dataset = ImagePathDataset(image_paths)
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=pil_collate,
    )
    if num_workers > 0:
        kwargs["persistent_workers"] = num_workers > 0 if persistent_workers is None else persistent_workers
        if prefetch_factor is not None:
            kwargs["prefetch_factor"] = prefetch_factor
    return DataLoader(**kwargs)


class VisionLanguageModel:
    def __init__(self, spec: ModelSpec, model: nn.Module, image_processor, device: str) -> None:
        self.spec = spec
        self.model = model
        self.image_processor = image_processor
        self.device = device
        self.model.eval()

    def pixel_values_from_images(self, images: Sequence[Image.Image]) -> torch.Tensor:
        tensors = [self.image_processor(img) for img in images]
        return torch.stack(tensors, dim=0)

    def image_input_dtype(self) -> torch.dtype:
        try:
            return next(self.model.parameters()).dtype
        except StopIteration:
            return torch.float32

    @torch.inference_mode()
    def encode_pixel_values(self, pixel_values: torch.Tensor, normalize: bool = False) -> torch.Tensor:
        target_dtype = self.image_input_dtype()
        if pixel_values.dtype != target_dtype and target_dtype in {torch.float16, torch.bfloat16, torch.float32}:
            pixel_values = pixel_values.to(dtype=target_dtype)
        pixel_values = pixel_values.to(self.device, non_blocking=True)
        feats = self.model.encode_image(pixel_values, normalize=normalize)
        return feats.float()

    @torch.inference_mode()
    def encode_images(self, images: Sequence[Image.Image], normalize: bool = False) -> torch.Tensor:
        pixel_values = self.pixel_values_from_images(images)
        return self.encode_pixel_values(pixel_values, normalize=normalize)


def load_model(model_key: str, device: str) -> VisionLanguageModel:
    if model_key not in MODEL_REGISTRY:
        known = ", ".join(sorted(MODEL_REGISTRY))
        raise KeyError(f"Unknown model key `{model_key}`. Known: {known}")

    import open_clip

    spec = MODEL_REGISTRY[model_key]
    model, _, preprocess = open_clip.create_model_and_transforms(
        spec.open_clip_model,
        pretrained=spec.open_clip_pretrained,
        device=device,
    )
    model.eval()
    return VisionLanguageModel(spec=spec, model=model, image_processor=preprocess, device=device)


@dataclass(frozen=True)
class EmbeddingRunConfig:
    batch_size: int
    num_workers: int
    checkpoint_every: int = 10
    use_fp16: bool = True
    resume: bool = True
    prefetch_factor: int = 2
    persistent_workers: bool = True


def load_embeddings_archive(path: Path) -> Dict[str, np.ndarray]:
    data = np.load(path)
    return {
        "embeddings": np.asarray(data["embeddings"], dtype=np.float32),
        "wnids": np.asarray(data["wnids"], dtype=str),
        "labels": np.asarray(data["labels"], dtype=str),
        "image_paths": np.asarray(data["image_paths"], dtype=str),
    }


def _save_archive(path: Path, **arrays) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(path, **arrays)


def _empty_device_cache(device: str) -> None:
    if device.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.empty_cache()
        return
    if device == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def compute_or_load_embeddings(
    vlm: VisionLanguageModel,
    image_paths: Sequence[str],
    class_map: Dict[str, str],
    output_dir: Path,
    config: EmbeddingRunConfig,
) -> Dict[str, np.ndarray]:
    batch_size = int(config.batch_size)
    num_workers = int(config.num_workers)
    while True:
        try:
            retry_config = EmbeddingRunConfig(
                batch_size=batch_size,
                num_workers=num_workers,
                checkpoint_every=config.checkpoint_every,
                use_fp16=config.use_fp16,
                resume=config.resume,
                prefetch_factor=config.prefetch_factor,
                persistent_workers=config.persistent_workers,
            )
            return _compute_or_load_embeddings_once(vlm, image_paths, class_map, output_dir, retry_config)
        except RuntimeError as exc:
            message = str(exc).lower()
            if num_workers > 0 and ("dataloader worker" in message or "exited unexpectedly" in message or "<stdin>" in message):
                print(f"[WORKERS] num_workers={num_workers} failed in this session; retrying with num_workers=0")
                num_workers = 0
                continue
            if ("out of memory" in message or "mps backend out of memory" in message or message.endswith("oom")) and batch_size > 32:
                next_batch_size = max(32, batch_size // 2)
                if next_batch_size == batch_size:
                    raise
                print(f"[OOM] batch_size={batch_size} failed on {vlm.device}; retrying with batch_size={next_batch_size}")
                _empty_device_cache(vlm.device)
                batch_size = next_batch_size
                continue
            raise


def _compute_or_load_embeddings_once(
    vlm: VisionLanguageModel,
    image_paths: Sequence[str],
    class_map: Dict[str, str],
    output_dir: Path,
    config: EmbeddingRunConfig,
) -> Dict[str, np.ndarray]:
    output_dir.mkdir(parents=True, exist_ok=True)
    final_path = output_dir / "embeddings.npz"
    checkpoint_path = output_dir / "embeddings_checkpoint.npz"

    image_paths_arr = np.asarray([str(p) for p in image_paths], dtype=str)
    wnids_arr = np.asarray([Path(p).parent.name for p in image_paths_arr], dtype=str)
    labels_arr = np.asarray([class_map.get(wnid, "unknown") for wnid in wnids_arr], dtype=str)

    if config.resume and final_path.exists():
        cached = load_embeddings_archive(final_path)
        if np.array_equal(cached["image_paths"], image_paths_arr):
            print(f"[CACHE] Loaded existing embeddings from {final_path.name}")
            return cached

    processed = 0
    embeddings = None
    if config.resume and checkpoint_path.exists():
        ckpt = np.load(checkpoint_path)
        ckpt_paths = np.asarray(ckpt["image_paths"], dtype=str)
        if np.array_equal(ckpt_paths, image_paths_arr):
            embeddings = np.asarray(ckpt["embeddings"], dtype=np.float32)
            processed = int(ckpt["processed"])
            print(f"[RESUME] Resuming from checkpoint at {processed}/{len(image_paths_arr)} images")

    loader = build_loader(
        image_paths_arr.tolist(),
        batch_size=config.batch_size,
        num_workers=config.num_workers,
        persistent_workers=config.persistent_workers,
        prefetch_factor=config.prefetch_factor,
        pin_memory=vlm.device != "cpu",
    )

    progress = None
    try:
        from tqdm.auto import tqdm

        progress = tqdm(total=len(image_paths_arr), initial=processed, desc="embeddings", unit="img")
    except Exception:
        progress = None

    t_start = time.perf_counter()
    try:
        for batch_idx, (images, batch_paths, _) in enumerate(loader):
            batch_start = batch_idx * config.batch_size
            batch_stop = batch_start + len(batch_paths)
            if batch_stop <= processed:
                continue

            skip = max(0, processed - batch_start)
            if skip:
                images = images[skip:]
                batch_paths = batch_paths[skip:]
                batch_start = processed

            pixel_values = vlm.pixel_values_from_images(images)
            if config.use_fp16:
                pixel_values = pixel_values.to(dtype=torch.float16)
            emb = vlm.encode_pixel_values(pixel_values, normalize=True).detach().cpu().to(torch.float32).numpy()

            if embeddings is None:
                embeddings = np.empty((len(image_paths_arr), emb.shape[1]), dtype=np.float32)

            batch_stop = batch_start + len(batch_paths)
            embeddings[batch_start:batch_stop] = emb
            processed = batch_stop

            if progress is not None:
                progress.update(len(batch_paths))

            if (batch_idx + 1) % config.checkpoint_every == 0 and processed < len(image_paths_arr):
                _save_archive(
                    checkpoint_path,
                    embeddings=embeddings,
                    wnids=wnids_arr,
                    labels=labels_arr,
                    image_paths=image_paths_arr,
                    processed=processed,
                )
    finally:
        if progress is not None:
            progress.close()

    if embeddings is None:
        raise RuntimeError("Embedding run produced no outputs.")

    elapsed = time.perf_counter() - t_start
    _save_archive(
        final_path,
        embeddings=embeddings,
        wnids=wnids_arr,
        labels=labels_arr,
        image_paths=image_paths_arr,
        processed=len(image_paths_arr),
    )
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print(f"[DONE] {embeddings.shape} in {elapsed:.1f}s ({len(image_paths_arr) / elapsed:.0f} img/s) -> {final_path.name}")
    return {
        "embeddings": embeddings,
        "wnids": wnids_arr,
        "labels": labels_arr,
        "image_paths": image_paths_arr,
    }


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "imagenet_val"
CLASS_MAP_PATH = PROJECT_ROOT / "data" / "imagenet_class_map.json"
OUTPUT_DIR = PROJECT_ROOT / "round3" / "part1_baseline_embeddings" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config
MODEL_KEY = "clip_b32"
REQUESTED_DEVICE = "auto"
DEVICE = resolve_device(REQUESTED_DEVICE)
USE_FP16 = use_fp16_for(DEVICE, requested=True)
NUM_IMAGES = 5000
BATCH_SIZE = default_batch_size(DEVICE)
NUM_WORKERS = default_num_workers()
CHECKPOINT_EVERY = 10
SEED = 42
RESUME = True

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device: {DEVICE} | fp16={USE_FP16} | batch_size={BATCH_SIZE} | workers={NUM_WORKERS}")

## Step 1: Preview Data

What does our dataset look like? Folder structure, label mapping, and actual sample images.

In [ ]:
# Load class map
with open(CLASS_MAP_PATH) as f:
    class_map = json.load(f)

# Show folder structure
class_dirs = sorted(d for d in DATA_DIR.iterdir() if d.is_dir())
print(f"{len(class_map)} classes, {len(class_dirs)} folders\n")
print("First 10 class folders:")
for d in class_dirs[:10]:
    n_imgs = len(list(d.glob("*.JPEG")))
    label = class_map.get(d.name, "???")
    print(f"  {d.name}/ -> {label:30s} ({n_imgs} images)")

In [ ]:
# Sample subset and show distribution
all_paths = list_images(str(DATA_DIR))
sampled = sample_paths(all_paths, NUM_IMAGES, SEED)
class_counts = Counter(Path(p).parent.name for p in sampled)
counts = sorted(class_counts.values())

print(f"Total images: {len(all_paths)}")
print(f"Sampled: {len(sampled)} across {len(class_counts)} classes")
print(f"Images per class: min={counts[0]}, max={counts[-1]}, median={counts[len(counts)//2]}")

# Show actual sample images
print(f"\nFirst 8 sampled images:")
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, p in zip(axes, sampled[:8]):
    with Image.open(p) as img:
        rgb = img.convert("RGB")
        label = class_map.get(Path(p).parent.name, "???")
        ax.imshow(rgb)
        ax.set_title(f"{label}\n{rgb.size[0]}x{rgb.size[1]}", fontsize=8)
        ax.axis("off")
plt.suptitle("Sample images from our subset", fontsize=12)
plt.tight_layout()
plt.show()

## Step 2: Load Model

Load CLIP ViT-B/32 and sanity-check it with a dummy image.

In [ ]:
t0 = time.time()
vlm = load_model(MODEL_KEY, DEVICE)
if USE_FP16:
    vlm.model.half()
model_dtype = vlm.image_input_dtype()

print(f"Model: {vlm.spec.open_clip_model} ({vlm.spec.open_clip_pretrained})")
print(f"Device: {vlm.device}")
print(f"Model dtype: {model_dtype}")
print(f"Load time: {time.time()-t0:.1f}s")

# Sanity check
test_img = Image.new("RGB", (224, 224), color=(128, 128, 128))
test_emb = vlm.encode_images([test_img], normalize=True)
print(f"\nSanity check (gray image):")
print(f"  Shape: {test_emb.shape}  (1 image, {test_emb.shape[1]}-dim)")
print(f"  L2 norm: {test_emb.norm().item():.4f}")
print(f"  First 10 dims: {test_emb[0, :10].cpu().numpy()}")

## Step 3: Compute Embeddings in Batches

Process all 5000 images. Save incrementally so we don't lose progress. Show example outputs along the way.

In [ ]:
run_config = EmbeddingRunConfig(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    checkpoint_every=CHECKPOINT_EVERY,
    use_fp16=USE_FP16,
    resume=RESUME,
)

result = compute_or_load_embeddings(
    vlm=vlm,
    image_paths=sampled,
    class_map=class_map,
    output_dir=OUTPUT_DIR,
    config=run_config,
)

embeddings = result["embeddings"]
wnids = result["wnids"]
labels = result["labels"]
all_img_paths = result["image_paths"]
final_path = OUTPUT_DIR / "embeddings.npz"

print(f"\nEmbeddings: {embeddings.shape} {embeddings.dtype}")
print(f"Saved: {final_path.name} ({final_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Show what we just computed: pick representative images across the run
show_idx = np.linspace(0, len(embeddings) - 1, num=min(5, len(embeddings)), dtype=int)
fig, axes = plt.subplots(len(show_idx), 2, figsize=(14, 2.5 * len(show_idx)), gridspec_kw={"width_ratios": [1, 3]})
axes = np.atleast_2d(axes)

for row, idx in enumerate(show_idx):
    with Image.open(all_img_paths[idx]) as img:
        rgb = img.convert("RGB")
        axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{labels[idx]}", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].bar(range(embeddings.shape[1]), embeddings[idx], width=1.0, color="steelblue", alpha=0.7)
    axes[row, 1].set_xlim(0, embeddings.shape[1])
    axes[row, 1].set_ylabel("value", fontsize=8)
    if row == len(show_idx) - 1:
        axes[row, 1].set_xlabel("embedding dimension")
    axes[row, 1].set_title(f"Embedding (norm={np.linalg.norm(embeddings[idx]):.4f})", fontsize=9)

plt.suptitle("Image -> Embedding examples", fontsize=13)
plt.tight_layout()
plt.show()

## Step 4: Analysis

Basic questions: Do same-class embeddings cluster? What does the embedding space look like?

In [ ]:
# Basic embedding stats
if "embeddings" not in globals():
    archive_path = OUTPUT_DIR / "embeddings.npz"
    if not archive_path.exists():
        raise FileNotFoundError("Run Step 3 first so embeddings.npz exists, then re-run Step 4.")
    archive = load_embeddings_archive(archive_path)
    embeddings = archive["embeddings"]
    wnids = archive["wnids"]
    labels = archive["labels"]
    all_img_paths = archive["image_paths"]
    print(f"Loaded embeddings from {archive_path.name}")

norms = np.linalg.norm(embeddings, axis=1)
print(f"Embedding norms: mean={norms.mean():.4f}, std={norms.std():.6f}")

# Pairwise cosine sim (sample up to 1000 images for speed)
rng = np.random.default_rng(SEED)
sample_size = min(1000, len(embeddings))
idx = rng.choice(len(embeddings), size=sample_size, replace=False)
subset = embeddings[idx]
cosines = subset @ subset.T
mask = ~np.eye(len(subset), dtype=bool)
off_diag = cosines[mask]

print(f"\nPairwise cosine similarity ({sample_size} random images):")
print(f"  mean={off_diag.mean():.4f}, std={off_diag.std():.4f}")
print(f"  min={off_diag.min():.4f}, max={off_diag.max():.4f}")

plt.figure(figsize=(8, 3))
plt.hist(off_diag, bins=100, alpha=0.7, color="steelblue")
plt.xlabel("Cosine similarity")
plt.ylabel("Count")
plt.title("Distribution of pairwise cosine similarities")
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# PCA
n_components = min(50, embeddings.shape[0], embeddings.shape[1])
if n_components < 3:
    raise ValueError("Need at least 3 samples/components for the PCA plots below.")

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(embeddings)
var_exp = pca.explained_variance_ratio_

print(f"Variance explained: PC1={var_exp[0]:.3f}, PC2={var_exp[1]:.3f}, PC3={var_exp[2]:.3f}")
print(f"Cumulative ({n_components} PCs): {var_exp.sum():.3f}")

# Scree plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, n_components + 1), var_exp, color="steelblue", alpha=0.7)
ax.set_xlabel("PC")
ax.set_ylabel("Variance explained")
ax2 = ax.twinx()
ax2.plot(range(1, n_components + 1), np.cumsum(var_exp), "r-o", markersize=3)
ax2.set_ylabel("Cumulative", color="red")
ax.set_title(f"PCA variance explained (CLIP B/32, {len(embeddings)} ImageNet images)")
plt.tight_layout()
plt.show()

In [ ]:
# PCA scatter — color 10 classes with most samples
unique_labels, label_counts = np.unique(labels, return_counts=True)
top10 = unique_labels[np.argsort(-label_counts)[:10]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, (pc_x, pc_y, lbl_x, lbl_y) in [
    (ax1, (0, 1, "PC1", "PC2")),
    (ax2, (0, 2, "PC1", "PC3")),
]:
    ax.scatter(X_pca[:, pc_x], X_pca[:, pc_y], c="lightgray", s=3, alpha=0.3)
    for cls in top10:
        m = labels == cls
        ax.scatter(X_pca[m, pc_x], X_pca[m, pc_y], s=12, alpha=0.7, label=cls)
    ax.set_xlabel(f"{lbl_x} ({var_exp[pc_x]:.1%})")
    ax.set_ylabel(f"{lbl_y} ({var_exp[pc_y]:.1%})")
    ax.legend(fontsize=7, loc="best", ncol=2)

fig.suptitle("CLIP B/32 embeddings — PCA colored by class", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Intra-class vs inter-class similarity
print("Intra-class cosine similarity (top 10 classes):\n")
intra_means = []
for cls in top10:
    m = labels == cls
    cls_emb = embeddings[m]
    if len(cls_emb) < 2:
        continue
    cos = cls_emb @ cls_emb.T
    triu = cos[np.triu_indices(len(cls_emb), k=1)]
    intra_means.append(triu.mean())
    print(f"  {cls:30s}  n={m.sum():2d}  cosine={triu.mean():.4f} +/- {triu.std():.4f}")

print(f"\n  Inter-class mean (random pairs): {off_diag.mean():.4f}")
print(f"  Intra-class mean (avg of above): {np.mean(intra_means):.4f}")
print(f"  Ratio: {np.mean(intra_means)/off_diag.mean():.2f}x tighter")